In [1]:
import pandas as pd

In [5]:
# 修改後的 Cell 2
# skiprows=2：跳過前兩列 
# header=0：將跳過後的下一列（第三列）當作標題欄位 
df_stations = pd.read_csv('../data_raw/114年文健站營運單位清冊.csv', skiprows=2, header=0)

# 清理欄位名稱（移除隱形的換行符號與空白） 
df_stations.columns = df_stations.columns.str.replace('\n', '').str.strip()

# 檢查一下欄位名稱是否正確出現
print("目前讀取到的欄位有：", df_stations.columns.tolist())

# 預覽前三筆資料確認對齊
df_stations.head(3)

目前讀取到的欄位有： ['編號', '縣市', '行政區', '名稱', '成立年度', '主要服務族群', '執行單位', '有無負責人', '計畫負責人', '地址', '電話', 'Unnamed: 11']


,編號,縣市,行政區,名稱,成立年度,主要服務族群,執行單位,有無負責人,計畫負責人,地址,電話,Unnamed: 11
0,1,臺北市,松山區,松山區,113,阿美族,社團法人台北市原住民關懷協會,有,林小鳳,臺北市松山區長春路339巷2號B1,02-25818669,NaN
1,2,新北市,汐止區,汐止區,107,阿美族,新北市原住民家庭關懷協會,有,張家豪,新北市汐止區樟樹二路302巷10號1樓(厚德山光市民活動中心),0921081155,NaN
2,3,新北市,新店區,溪洲,111,阿美族,社團法人新北市新店區溪洲阿美族文化永續發展協會,有,張祖淼,新北市新店區溪洲路 250號,0916818637,NaN


In [3]:
df_stations.head()

,原住民族委員會文化健康站營運單位清冊,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,114.8.8,NaN
1,編號,縣市,行政區,名稱,成立年度,主要服務\n族群,執行單位,有無\n負責人,計畫\n負責人,地址,電話,NaN
2,1,臺北市,松山區,松山區,113,阿美族,社團法人台北市原住民關懷協會,有,林小鳳,臺北市松山區長春路339巷2號B1,02-25818669,NaN
3,2,新北市,汐止區,汐止區,107,阿美族,新北市原住民家庭關懷協會,有,張家豪,新北市汐止區樟樹二路302巷10號1樓(厚德山光市民活動中心),0921081155,NaN
4,3,新北市,新店區,溪洲,111,阿美族,社團法人新北市新店區溪洲阿美族文化永續發展協會,有,張祖淼,新北市新店區溪洲路 250號,0916818637,NaN


In [6]:
# 修改後的 Cell 4
# 1. 確保成立年度是數字格式 
df_stations['成立年度'] = pd.to_numeric(df_stations['成立年度'], errors='coerce')

# 2. 移除沒有成立年度或資料異常的列 
df_clean = df_stations.dropna(subset=['成立年度']).copy()

# 3. 依照縣市、行政區與年度統計「新成立」數量 
# size() 會計算每個分組的數量，unstack() 會把年度從直向轉為橫向 
town_yearly_new = df_clean.groupby(['縣市', '行政區', '成立年度']).size().unstack(fill_value=0)

# 4. 關鍵步驟：計算「累積站數」 
# cumsum(axis=1) 代表橫向累加，這能呈現該鄉鎮在該年度的總服務量 
town_panel = town_yearly_new.cumsum(axis=1)

# 5. 顯示結果
print("各鄉鎮歷年累積站數面板資料：")
town_panel

各鄉鎮歷年累積站數面板資料：


成立年度      90   95   96   97   98   99   100  101  102  103  104  105  106  \
縣市  行政區                                                                     
南投縣 仁愛鄉     1    2    3    3    3    3    3    4    5    5    5    5    8   
    信義鄉     1    3    3    4    5    5    5    5    5    5    6    6    7   
    南投市     0    0    0    0    0    0    0    0    0    0    0    0    0   
    埔里鎮     0    0    0    0    0    0    0    0    0    0    0    1    1   
    魚池鄉     0    0    0    0    0    0    0    0    0    0    0    0    0   
...       ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
高雄市 桃源區     0    0    0    0    1    1    1    1    1    1    1    1    4   
    楠梓區     0    0    0    0    0    0    0    0    0    0    0    0    1   
    茂林區     0    0    0    0    1    1    1    1    2    2    2    2    3   
    那瑪夏區    0    0    0    0    0    0    0    1    1    1    1    1    2   
    鳳山區     0    0    0    0    0    0    0    0    0    0    0    0    1   

成立年度      107  108  109  111  112  113  114  
縣市  行政區                                      
南投縣 仁愛鄉    11   13   19   21   21   21   21  
    信義鄉     8    9   10   11   11   11   11  
    南投市     0    0    1    1    1    1    1  
    埔里鎮     1    2    3    3    4    4    4  
    魚池鄉     1    1    1    1    1    1    1  
...       ...  ...  ...  ...  ...  ...  ...  
高雄市 桃源區     4    6    8    8    8    8    8  
    楠梓區     1    1    1    1    1    1    1  
    茂林區     3    3    3    3    3    3    3  
    那瑪夏區    3    3    3    3    3    3    3  
    鳳山區     1    1    1    1    1    1    1  

[122 rows x 20 columns]

In [7]:
# 存成 CSV 檔，encoding='utf_8_sig' 可確保 Excel 開啟時中文不亂碼
town_panel.to_csv('../data_processed/town_yearly_panel.csv', encoding='utf_8_sig')